# MonoDGP M59b: Core ML intermediate-tensor diagnostic

Run every cell on a Colab GPU with high RAM. This notebook does not train or change weights. It reuses the exact M58/M57 provenance, exports a separate FP32 diagnostic package, and stops for macOS execution. The package is designed to identify the first tensor that diverges after M58 macOS parity failed.

Read `MONODGP_M59B_COREML_DIAGNOSTIC_CONTRACT.md` before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import hashlib, json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d')
MONODGP_REPO=Path('/content/MonoDGP_M59b')
MONODGP_COMMIT='aa059a18214aebf644510e7f0793971b403f9d14'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodgp_kitti_m56d')
M57_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m57_deformable_attention')
M57_MANIFEST=M57_ROOT/'m57_deformable_attention_manifest.json'
M57_SMOKE=M57_ROOT/'m57_deformable_attention_smoke.json'
M57_GATE=M57_ROOT/'complete_evaluation/m57_portable_attention_gate.json'
M57_COMPARISON=M57_ROOT/'complete_evaluation/m57_portable_attention_comparison.csv'
M58_GATE=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m58_coreml_conversion/m58_coreml_export_gate.json')
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m59b_coreml_diagnostic')
LOG_DIR=OUTPUT_ROOT/'colab_logs'
REPORT=OUTPUT_ROOT/'m59b_coreml_diagnostic_export_gate.json'
def sha256(path):
 digest=hashlib.sha256()
 with Path(path).open('rb') as handle:
  for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
 return digest.hexdigest()
def run(command,cwd=None,env=None):
 command=[str(item) for item in command]; print('+',shlex.join(command),flush=True)
 merged=os.environ.copy(); merged.update(env or {})
 result=subprocess.run(command,cwd=cwd,env=merged)
 if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_logged(command,cwd,log_path,env=None):
 command=[str(item) for item in command]; print('+',shlex.join(command),flush=True)
 log_path=Path(log_path); log_path.parent.mkdir(parents=True,exist_ok=True); merged=os.environ.copy(); merged.update(env or {}); tail=deque(maxlen=120)
 with log_path.open('w',encoding='utf-8') as log:
  process=subprocess.Popen(command,cwd=cwd,env=merged,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
  for line in process.stdout:
   print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
  code=process.wait()
 if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
 return log_path
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
run(['nvidia-smi'])

In [ ]:
# Restore the exact M58 source path and portable deformable-attention patch.
if not MOBILE_REPO.exists(): run(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO])
else: run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODGP_REPO.exists(): run(['git','clone','https://github.com/PuFanqi23/MonoDGP.git',MONODGP_REPO])
run(['git','fetch','--all'],cwd=MONODGP_REPO)
# This Colab checkout is disposable; clear patches from an interrupted/prior run.
run(['git','reset','--hard',MONODGP_COMMIT],cwd=MONODGP_REPO)
run(['git','checkout',MONODGP_COMMIT],cwd=MONODGP_REPO)
run([sys.executable,'-m','pip','install','-q','coremltools==9.0','pyyaml','scipy','opencv-python-headless','numba','scikit-image','scikit-learn','tqdm','ninja','pandas'])
for script in ('patch_monodgp_colab_compat.py','patch_monodgp_m54_training.py','patch_monodgp_m57_deformable_attention.py','patch_monodgp_m58_coreml.py'):
 run([sys.executable,f'scripts/{script}','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
tap_path=MOBILE_REPO/'scripts/export_monodgp_m59b_coreml_diagnostics.py'
tap_text=tap_path.read_text()
tap_text=tap_text.replace('self.taps[f"final_{key}"] = values[key]','self.taps.put(f"final_{key}", values[key])').replace('self.taps[f"final_region_prob_{index}"] = value','self.taps.put(f"final_region_prob_{index}", value)')
if 'self.taps.put(f"final_{key}"' not in tap_text: raise RuntimeError('M59b TapStore compatibility fix was not applied')
tap_path.write_text(tap_text)
print('M59b TapStore writer compatibility verified')
ops=MONODGP_REPO/'lib/models/monodgp/ops'; shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, coremltools, MultiScaleDeformableAttention; print(torch.__version__,torch.version.cuda,coremltools.__version__)'],cwd=MONODGP_REPO)

In [ ]:
# Verify the frozen M58 gate and the four reviewed M57 artifacts.
if not M58_GATE.is_file(): raise FileNotFoundError(M58_GATE)
m58=json.loads(M58_GATE.read_text())
if not (m58.get('complete') and m58.get('all_export_gates_passed')): raise RuntimeError(m58)
REVIEWED={M57_MANIFEST:'7c4757798dfff4d95053912730faff4c5d7a4626ff41a539ad87b2f9193eb313',M57_SMOKE:'2a96cffd8e1e6b77f2c547c2b94dca4bde2e72bf4185d853ee7bca71ed28b3b0',M57_GATE:'63f2b71d1f5be69bad31907db229f79e39fb17c033ba43999c6fdfe1df3b97a7',M57_COMPARISON:'687b15cd4a044981f4e00fa038f2fc5e7053cee692e13bfdb85fe2f90656412d'}
for path,expected in REVIEWED.items():
 if not path.is_file(): raise FileNotFoundError(path)
 if sha256(path)!=expected: raise RuntimeError(f'Reviewed M57 evidence changed: {path}')
def resolve(root,names):
 for name in names:
  path=root/name
  if path.is_dir(): return path
sources={key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names) for key,names in {'image_2':['training/image_2','training/image_02'],'label_2':['training/label_2','training/label_02'],'calib':['training/calib']}.items()}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True); (DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
 link=DATASET_ROOT/'training'/name
 if link.is_symlink() and link.resolve()==target.resolve(): continue
 if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
 link.symlink_to(target,target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769
print('M58/M57 provenance and Chen split verified; M59b diagnostic export authorized.')

In [ ]:
# Export the separate intermediate-tensor package.
EXPORT_LOG=run_logged([sys.executable,'-u','scripts/export_monodgp_m59b_coreml_diagnostics.py','--monodgp-repo',MONODGP_REPO,'--m58-export-gate',M58_GATE,'--m57-manifest',M57_MANIFEST,'--m57-smoke',M57_SMOKE,'--m57-gate',M57_GATE,'--m57-comparison',M57_COMPARISON,'--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR,'--output-dir',OUTPUT_ROOT],MOBILE_REPO,LOG_DIR/'m59b_diagnostic_export.log',env={'MONODGP_PORTABLE_DEFORM_ATTN':'1'})
report=json.loads(REPORT.read_text())
if not report.get('complete') or not report.get('all_export_gates_passed'): raise RuntimeError(report)
if report.get('physical_device_testing_authorized') or report.get('fp16_or_quantization_authorized') or report.get('product_safety_qualified'): raise RuntimeError(report)
print(json.dumps(report,indent=2)); print('STOP: copy the three M59b artifacts to macOS and run the validator.')

## Stop point

Return `m59b_coreml_diagnostic_export_gate.json`, `m59b_diagnostic_reference_io.npz`, the `.mlpackage` (or ZIP), and the durable log. Then run `scripts/validate_monodgp_m59b_macos_diagnostics.py` on macOS. Do not run an iPhone, FP16, quantization, or deployment test until the first divergence is understood.